# Predicting Mobile Game "Whales": Free-to-Play Payer Conversion

**Business problem:** In free-to-play mobile games, only a small fraction of players ever spend money (often called "whales" when they spend a lot). Studios want to predict, from early player behavior, **which players are likely to convert into paying customers**,so they can target them with personalized offers, special deals, or push notifications.

**Task:** Binary Classification i.e. predict `converted_to_payer` (1 = converts to a paying user, 0 = stays free-to-play).

---
### Project Roadmap
1. Load Dataset
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Building: Logistic Regression, KNN, SVM, Decision Tree, Bagging, Random Forest, AdaBoost, Gradient Boosting, Voting Classifier
5. Model Evaluation: Accuracy, Precision, Recall, F1, ROC-AUC
6. Model Comparison

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Dataset

In [ ]:
df = pd.read_csv("mobile_game_whale_dataset.csv")
print("Dataset shape:", df.shape)
df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Basic Info & Summary Statistics

In [ ]:
df.info()

In [ ]:
df.describe()

### 3.2 Missing Values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing)

plt.figure(figsize=(6,4))
sns.barplot(x=missing.index, y=missing.values, color='indianred')
plt.title("Missing Values per Column")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 3.3 Target Variable Distribution

In [ ]:
target_counts = df['converted_to_payer'].value_counts(normalize=True) * 100
print(target_counts)

plt.figure(figsize=(5,4))
sns.countplot(x='converted_to_payer', data=df)
plt.title("Target Distribution: Converted to Payer")
plt.xlabel("Converted to Payer (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.show()

This is an **imbalanced classification problem**. Most players never become payers, which is realistic for this domain.

### 3.4 Numerical Feature Distributions

In [ ]:
num_cols_plot = ['age', 'sessions_last_7d', 'total_playtime_hours', 'levels_completed',
                  'store_visits', 'wishlist_items', 'streak_days', 'level_fail_rate']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols_plot):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

### 3.5 Feature Relationships with the Target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
compare_cols = ['sessions_last_7d', 'total_playtime_hours', 'store_visits',
                 'wishlist_items', 'streak_days', 'rage_quit_events']
for i, col in enumerate(compare_cols):
    sns.boxplot(x='converted_to_payer', y=col, data=df, ax=axes[i])
    axes[i].set_title(f"{col} vs Converted")
plt.tight_layout()
plt.show()

### 3.6 Categorical Features vs Target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x='acquisition_channel', y='converted_to_payer', data=df, ax=axes[0])
axes[0].set_title("Conversion Rate by Acquisition Channel")
axes[0].set_ylabel("Conversion Rate")

sns.barplot(x='device_type', y='converted_to_payer', data=df, ax=axes[1])
axes[1].set_title("Conversion Rate by Device Type")
axes[1].set_ylabel("Conversion Rate")
plt.tight_layout()
plt.show()

### 3.7 Correlation Heatmap (Numerical Features)

In [ ]:
plt.figure(figsize=(14, 10))
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['player_id'])
corr = round(numeric_df.corr(),2)
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True)
plt.title("Correlation Heatmap")
plt.show()

## 4. Data Preprocessing

Steps:
1. Drop identifier column (`player_id`)
2. Separate features (`X`) and target (`y`)
3. Train-test split (stratified, since the target is imbalanced)
4. Handle missing values (median for numeric, mode for categorical)
5. Encode categorical variables (One-Hot Encoding)
6. Feature scaling (needed for Logistic Regression, KNN, SVM i.e. distance/gradient based algorithms)

We'll use a `ColumnTransformer` + `Pipeline` so every model gets **identical, leak-free preprocessing** (fit only on training data, applied to test data).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
df.drop("player_id",axis=1,inplace=True)

In [ ]:
target = "converted_to_payer"
X = df.drop(columns=target)
y = df[target]

In [ ]:
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target distribution:\n", y_train.value_counts(normalize=True))
print("\nTest target distribution:\n", y_test.value_counts(normalize=True))

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

preprocessor

In [ ]:
cat_cols

## 5. Model Building & Evaluation

For each algorithm we:
1. Build a `Pipeline` = preprocessing + model
2. Fit on the training set
3. Predict on the test set
4. Report **Accuracy, Precision, Recall, F1-score, ROC-AUC**, plus a confusion matrix

We'll store every model's metrics in a `results` list so we can compare them all at the end.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)

results = []          # to store metrics for final comparison
fitted_models = {}    # to store fitted pipelines for ROC curve comparison

def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]


    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba)

    print(f"===== {name} =====")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"ROC-AUC   : {auc:.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Not Converted', 'Converted'],
                yticklabels=['Not Converted', 'Converted'])
    plt.title(f"Confusion Matrix - {name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()

    results.append({
        'Model': name, 'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1 Score': f1, 'ROC-AUC': auc
    })
    fitted_models[name] = pipeline
    return pipeline

### 5.1 Logistic Regression

In [ ]:
X_test

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

evaluate_model("Logistic Regression", logreg_pipeline, X_train, y_train, X_test, y_test)


### 5.2 K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

evaluate_model("KNN", knn_pipeline, X_train, y_train, X_test, y_test)


### 5.3 Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC

svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE))
])

evaluate_model("SVM", svm_pipeline, X_train, y_train, X_test, y_test)


### 5.4 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

evaluate_model("Decision Tree", dt_pipeline, X_train, y_train, X_test, y_test)


### 5.5 Bagging Classifier

In [ ]:
from sklearn.ensemble import BaggingClassifier

bagging_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
                                 n_estimators=50, random_state=RANDOM_STATE))
])

evaluate_model("Bagging (Decision Trees)", bagging_pipeline, X_train, y_train, X_test, y_test)


### 5.6 Random Forest (Bagging Ensemble)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE))
])

evaluate_model("Random Forest", rf_pipeline, X_train, y_train, X_test, y_test)


### 5.7 AdaBoost (Boosting)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

adaboost_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE))
])

evaluate_model("AdaBoost", adaboost_pipeline, X_train, y_train, X_test, y_test)


### 5.8 Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))
])

evaluate_model("Gradient Boosting", gb_pipeline, X_train, y_train, X_test, y_test)


### 5.9 Voting Classifier (combining multiple models)

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
            ('rf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
            ('dt', DecisionTreeClassifier(random_state=RANDOM_STATE))
        ],
        voting='soft'
    ))
])

evaluate_model("Voting Classifier", voting_pipeline, X_train, y_train, X_test, y_test)


## 6. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True)
results_df

In [ ]:
results_melted = results_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(14, 6))
sns.barplot(data=results_melted, x='Model', y='Score', hue='Metric')
plt.title("Model Comparison Across Metrics")
plt.xticks(rotation=30, ha='right')
plt.ylim(0, 1)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 6.1 ROC Curve Comparison

In [ ]:
plt.figure(figsize=(9, 7))
ax = plt.gca()
for name, pipeline in fitted_models.items():
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, ax=ax, name=name)

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
plt.title("ROC Curve Comparison of All Models")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Observations

- Take note of which models have **high accuracy but low recall**, with only ~13% of players converting, a model can score well on accuracy just by predicting "no conversion" most of the time. That is why **accuracy alone is misleading for imbalanced data**, and Precision/Recall/F1/ROC-AUC matter more here.
- Tree-based ensembles (Random Forest, Gradient Boosting, AdaBoost) generally handle the non-linear engagement patterns better than a plain Decision Tree or KNN.
- All models above were evaluated using their **default hyperparameters** and a **single train/test split**, so these numbers are only a first look.

---
## 8. Cross-Validation & Hyperparameter Tuning

Topics to cover next:
- K-Fold Cross-Validation (`cross_val_score`, `StratifiedKFold`)
- `GridSearchCV` / `RandomizedSearchCV` for each model
- Comparing cross-validated performance vs. the single-split results above
- Selecting final tuned models and re-evaluating on the test set


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Define the final pipeline with Logistic Regression (default parameters, or tuned if you wish)
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# Evaluate with cross-validation (optional, but recommended)
cv_scores_final = cross_val_score(final_model, X_train, y_train, cv=cv, scoring='roc_auc')
print(f"Cross-validated ROC-AUC for Logistic Regression: {cv_scores_final.mean():.4f} (+/- {cv_scores_final.std():.4f})")

# Fit the final model on the entire training set
final_model.fit(X_train, y_train)

In [ ]:
import joblib
import os

# Ensure the directory exists
os.makedirs("models", exist_ok=True)

# Save the final pipeline
joblib.dump(final_model, "models/whale_prediction_model.pkl")
print("Model saved to models/whale_prediction_model.pkl")

In [36]:
pip install streamlit

   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
    --------------------------------------- 0.3/10.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.5 MB 1.0 MB/s eta 0:00:10
   -- ------------------------------------- 0.8/10.5 MB 931.2 kB/s eta 0:00:11
   -- ------------------------------------- 0.8/10.5 MB 931.2 kB/s eta 0:00:11
   --- ------------------------------------ 1.0/10.5 MB 898.8 kB/s eta 0:00:11
   ---- ----------------------------------- 1.3/10.5 MB 871.6 kB/s eta 0:00:11
   ---- ----------------------------------- 1.3/10.5 MB 871.6 kB/s eta 0:00:11
   ----- ---------------------------------- 1.6/10.5 MB 830.6 kB/s eta 0:00:11
   ------ --------------------------------- 1.8/10.5 MB 824.8 kB/s eta 0:00:11
   ------- -------------------------------- 2.1/10.5 MB 869.7 kB/s eta 0:00:10
   ------- -------------------------------- 2.1/10.5 MB 869.7 kB/s eta 0:00:10
  

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 7.35.1 which is incompatible.
tensorflow-intel 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 7.35.1 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [ ]:
!Streamlit run app.py